# PS1: Your first library-free neural network!  

Advanced Learning 2026


For SUBMISSION:   

Please upload the complete and executed `ipynb` to your git repository. Verify that all of your output can be viewed directly from github, and provide a link to that git file below.

~~~
STUDENT ID: 207600495
~~~

~~~
STUDENT GIT LINK: https://github.com/YonatanSas/52025-Adv-computational-learning-and-data-analysis/blob/main/assignment_1/PS1_Part2_Hello_NN_2026_ID_207600495.ipynb
~~~
In Addition, don't forget to add your ID to the files:    
  
`PS1_Part2_Hello_NN_2026_ID_207600495.html`   


In [1]:
import numpy as np # You are allowed to use  only numpy.
import time


**Welcome**.   

In this part of the problem set you are set to build a complete and flexible neural network.  
This neural network will be library free (in the sense that we won't use PyTorch/Tensorflow/etc.).   

Let's do a quick review of the basic neural-network components:  


*   *Layer* - can be fully connected (dense/hidden), convolution, etc.
  * Forward propagation- the layer outputs the next layer's input
  * Backward propagation- the layer also outputs the gradient descent update
*   *Activation* Layer (e.g. ReLU) - there are no parameters, only gradients with respect to the input. We want to compute both the gradient w.r.t the parameters of the layer and to create the gradient with respect to the layer's inputs
   * *Forward propagation*- the layer outputs the next layer's input
   * *Backward propagation*- the layer also outputs the gradient descent update
*   *Loss Function* : how our model  quantifies the difference between the predicted outputs the actual (target) values  
*   *Network Wrapper*-  wraps our components together as a trainable model.






Useful resource:  
* Gradient descent for neural networks [cheat sheet](https://moodle4.cs.huji.ac.il/hu23/mod/resource/view.php?id=402297).
* Neural network architecture [cheat sheet](https://moodle4.cs.huji.ac.il/hu23/mod/url/view.php?id=402298).

### 0. Loading data

You are going to test and evaluate your home-made network on the `mnist` dataset.   
The MNIST dataset is a large dataset of handwritten digits that is commonly used for training various image and vision models.

In [2]:
from keras.datasets import mnist
from keras.utils import to_categorical
# load MNIST from server
# Using a standard library (keras.datasets) to load the mnist data
(x_train, y_train), (x_test, y_test) = mnist.load_data()

#### Data transformations





In [3]:
# training data : 60000 samples
# reshape and normalize input data
x_train = x_train.reshape(x_train.shape[0], 1, 28*28)
x_train = x_train.astype('float32')
x_train /= 255

# One-hot encoding of the output.
# Currently a number in range [0,9]; Change into a vector of size 10
# e.g. number 3 will become [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train = to_categorical(y_train)

# same for test data : 10000 samples
x_test = x_test.reshape(x_test.shape[0], 1, 28*28)
x_test = x_test.astype('float32')
x_test /= 255
y_test = to_categorical(y_test)


### 1. Network's Components

Please fill-in the missing code in the code boxes below (only where  `#### SOLUTION REQUIRED ####` is specified).   

In [4]:
# This class is a general layer primitive, defining that each instance must
# have an (input,output) parameters, and 2 functions: forward+backward propogation
class Layer_Primitive:
    def __init__(self):
        self.input = None
        self.output = None

    # computes the output Y of a layer for a given input X
    def forward_propagation(self, input):
        raise NotImplementedError

    # computes dE/dX for a given dE/dY (and update parameters if any)
    def backward_propagation(self, output_error, learning_rate):
        raise NotImplementedError


#### Fully Connected Layer

A fully-connected layer (a.k.a. affine, dense,linear layer) connects every input neuron to every output neuron.   
It has 2 parameters: (input, output).   
You need to define (code) the following:
* its initialization weights with random weights.
* the forward propogation calculation (as shown in class).
* the backward propogation gradients calculation (given output, as shown in class).

Parameters must be intitialized with some values. There are many ways to initialize the weights, and you are encouraged to do a quick research about the common methods. Any commonly used method will be accepted.  

1.1 (20 pts)

In [5]:
#### SOLUTION REQUIRED ####

# inherit from base class Layer
class Affine_Layer(Layer_Primitive):
    # input_size = number of input neurons
    # output_size = number of output neurons
    def __init__(self, input_size, output_size):
        # self.weights = #  # FILL IN THE MISSING CODE: randomly initialize the weights of the input_size, output_size size layer.
        self.weights = np.random.randn(input_size, output_size) * 0.1 # random so the neurons are different from each other, small so the outputs dont blow up
        # self.bias = # # FILL IN THE MISSING CODE: randomly initialize the bias of the  layer.
        self.bias = np.random.randn(1, output_size) * 0.1# bias - same idea, shape (1, output_size) so it matches the output row


    # returns output for a given input
    def forward_propagation(self, input_data):
        self.input = input_data
        self.output = np.dot(self.input, self.weights) + self.bias # Y = XW + b
        return self.output

    # computes dE/dW, dE/dB for a given output_error=dE/dY. Returns input_error=dE/dX.
    def backward_propagation(self, output_grad, learning_rate):
        input_error = np.dot(output_grad, self.weights.T)   # dE/dX = dE/dY * W^T
        weights_error = np.dot(self.input.T, output_grad)   # dE/dW = X^T * dE/dY
        # its like part 1

        # update parameters
        self.weights -= learning_rate * weights_error
        self.bias -= learning_rate * output_grad

        return input_error


#### Activation layers

Activation functions are often a non-linear functions that aid in how well the network model adapts to and learns  the training dataset. The choice of activation function in the output layer will define the type of predictions the model can make.  



In [6]:
# inherit from base class Layer
class ActivationLayer(Layer_Primitive):
    def __init__(self, activation, activation_grad):
        self.activation = activation
        self.activation_grad = activation_grad

    # returns the activated input
    def forward_propagation(self, input_data):
        self.input = input_data
        self.output = self.activation(self.input)
        return self.output

    # Returns input_error=dE/dX for a given output_grad=dE/dY.
    # learning_rate is not used because there is no "learnable" parameters.
    def backward_propagation(self, output_grad, learning_rate):
        return self.activation_grad(self.input) * output_grad



You need to define (code) the following via different functions:
* the forward propogation calculation (as shown in class).
* the backward propogation gradients calculation (given output, as shown in class).

1.2 (20 pts)

In [7]:
#### SOLUTION REQUIRED ####

# activation functions and their derivatives:

def tanh(x):
    return np.tanh(x)

def tanh_grad(x):
    return 1 - np.tanh(x)**2

def relu(x):
    return np.maximum(0, x)

def relu_grad(x):
    return (x > 0).astype(float)

def sigmoid(x):
    # FILL IN THE MISSING CODE
    x = np.clip(x, -500, 500)   # so exp wont overflow
    return 1 / (1 + np.exp(-x))

def sigmoid_grad(x):
    # FILL IN THE MISSING CODE
    s = sigmoid(x)
    return s * (1 - s)


#### Loss function

1.3 (10 pts)

In [8]:
#### SOLUTION REQUIRED ####

# loss function and its derivative

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

def mse_grad(y_true, y_pred):
    return 2 * (y_pred - y_true) / y_true.size
    # d/dy_pred of mean((y_true-y_pred)^2) = 2*(y_pred-y_true)/n


#### Putting everything together

1.4 (10 pts)

In [9]:
#### SOLUTION REQUIRED (in `predict`) ####

class MyNetwork:
    def __init__(self):
        self.layers = []
        self.loss = None
        self.loss_grad = None

    # add layer to network
    def add(self, layer):
        self.layers.append(layer)

    # set loss to use
    def use_loss(self, loss, loss_grad):
        self.loss = loss
        self.loss_grad = loss_grad


    # train the network
    def fit(self, x_train, y_train, epochs, learning_rate):
        # sample dimension first
        samples = len(x_train)

        # training loop
        for i in range(epochs):
            err = 0
            for j in range(samples):
                # forward propagation
                output = x_train[j]
                for layer in self.layers:
                    output = layer.forward_propagation(output)

                # compute loss (for display purpose only)
                err += self.loss(y_train[j], output)

                # backward propagation
                grad = self.loss_grad(y_train[j], output)
                for layer in reversed(self.layers):
                    grad = layer.backward_propagation(grad, learning_rate)

            # calculate average error on all samples
            err /= samples
            print('Training epoch %d/%d   error=%f' % (i+1, epochs, err))


    # predict output for given input
    def predict(self, x_test,y_test=np.array([])):
        if y_test.size:
           assert len(x_test)==len(y_test) # if Y is given
        # sample dimension first
        samples = len(x_test)
        result = []
        loss = 0
        correct = 0
        # run network over all samples
        for i in range(samples):
            # forward propagation
            output = x_test[i]
            for layer in self.layers:
                output = layer.forward_propagation(output)
            result.append(output)
            # ONLY IF LABELS ARE GIVEN (Y):
            if y_test.size:
                # Evaluate the output against Y,
                # calculate loss against Y, add to `loss`:
                loss += self.loss(y_test[i], output)
                target = y_test[i]
                # Evaluate the label of the output against real, and if identical,
                # add +1 to `correct`:
                if np.argmax(output) == np.argmax(target):
                   correct += 1
        if y_test.size:
            mean_loss = loss/samples

            print('\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.
                  format(mean_loss, correct, samples,100. * correct / samples))

        return result


## 2. Testing Your Neural Network

### Defining our main neural network architecture

Define your network's architecture:  
(Please rationalize your choice of activation funciton.)
* first affine layer that takes your input and outputs 128 nodes
* `tanh/relu/sigmoid` activation layer following the first affine layer
* second affine layer that takes the first layer's input and outputs 64 nodes
* `tanh/relu/sigmoid` activation layer following the second affine layer
* third affine layer that takes your second layer's input and outputs nodes in the size of the Y labels.
* `tanh/relu/sigmoid` activation layer following the last affine layer


2.1 (5 pts)

In [10]:
#### SOLUTION REQUIRED (in `predict`) ####
# i wrote an explanation after this cell why i chose these
# Network Architecture
np.random.seed(123)# so I get the same results when I run it again...
net = MyNetwork()
net.add(Affine_Layer(28*28, 128))# 784 to 128
net.add(ActivationLayer(relu, relu_grad))
net.add(Affine_Layer(128, 64)) # 128 to 64
net.add(ActivationLayer(relu, relu_grad))
net.add(Affine_Layer(64, 10)) # 64 to 10 (number of classes)
net.add(ActivationLayer(tanh, tanh_grad))


In [11]:
# quick test to choose the activations
def try_net(hidden, hidden_grad, last, last_grad):
    test_net = MyNetwork()
    test_net.add(Affine_Layer(28*28, 128))
    test_net.add(ActivationLayer(hidden, hidden_grad))
    test_net.add(Affine_Layer(128, 64))
    test_net.add(ActivationLayer(hidden, hidden_grad))
    test_net.add(Affine_Layer(64, 10))
    test_net.add(ActivationLayer(last, last_grad))
    test_net.use_loss(mse, mse_grad)
    test_net.fit(x_train[:10000], y_train[:10000], epochs=2, learning_rate=0.05)
    # check on the last 10,000 train images (validation), not on the test set
    # (predict still prints "Test set", but here it is the validation part)
    test_net.predict(x_train[50000:], y_train[50000:])

for run in range(2): # run it twice because the starting weights are random
    print("relu, relu, tanh:")
    try_net(relu, relu_grad, tanh, tanh_grad)
    print("relu, relu, sigmoid:")
    try_net(relu, relu_grad, sigmoid, sigmoid_grad)
    print("relu, relu, relu:")
    try_net(relu, relu_grad, relu, relu_grad)
    print("tanh, tanh, tanh:")
    try_net(tanh, tanh_grad, tanh, tanh_grad)
    print("sigmoid, sigmoid, sigmoid:")
    try_net(sigmoid, sigmoid_grad, sigmoid, sigmoid_grad)

relu, relu, tanh:
Training epoch 1/2   error=0.034084
Training epoch 2/2   error=0.016773

Test set: Avg. loss: 0.0161, Accuracy: 9167/10000 (92%)

relu, relu, sigmoid:
Training epoch 1/2   error=0.042749
Training epoch 2/2   error=0.017567

Test set: Avg. loss: 0.0165, Accuracy: 9053/10000 (91%)

relu, relu, relu:
Training epoch 1/2   error=0.043571
Training epoch 2/2   error=0.026624

Test set: Avg. loss: 0.0237, Accuracy: 8509/10000 (85%)

tanh, tanh, tanh:
Training epoch 1/2   error=0.040804
Training epoch 2/2   error=0.023705

Test set: Avg. loss: 0.0206, Accuracy: 9186/10000 (92%)

sigmoid, sigmoid, sigmoid:
Training epoch 1/2   error=0.090124
Training epoch 2/2   error=0.086894

Test set: Avg. loss: 0.0840, Accuracy: 2890/10000 (29%)

relu, relu, tanh:
Training epoch 1/2   error=0.034689
Training epoch 2/2   error=0.016848

Test set: Avg. loss: 0.0159, Accuracy: 9183/10000 (92%)

relu, relu, sigmoid:
Training epoch 1/2   error=0.042403
Training epoch 2/2   error=0.016870

Test s

so why I chose these activation functions?

Hidden layers - ReLU. Each ReLU unit is an "elbow" function, and enough elbows can approximate any continuous function as we learned. Without a non linear activation, all the layers collapse into one linear map. ReLU is also cheap to compute, and its derivative is just 0 or 1.

Output layer - tanh. The targets are one-hot vectors (nine 0's and one 1) and the loss is MSE, so I want a bounded output. For tanh, 0 is in the middle of the function, where the slope is the largest. A sigmoid gets close to 0 only in its flat part, where the derivative is almost 0, so very little gradient flows back (like the "dead zone" of an elbow).

The quick test above (each option was trained on 10,000 images for 2 epochs and checked on the last 10,000 training images as a validation set, not on the test set). Sigmoid in all the layers was far behind the others (around 30%),
because the gradient gets smaller in every sigmoid layer it passes through. ReLU in the hidden layers with tanh at the output was among the best (about 92% in both runs). tanh in all the layers got about the same, so for the hidden
layers the test does not decide, and I kept ReLU for the reasons above. A ReLU output was much worse and not stable (85% and 76% in the two runs): it is not bounded, and its derivative is 0 for negative values, so an output unit that
falls into the dead zone of the elbow stops learning. A sigmoid output was a bit lower than tanh (about 91%), so I kept tanh at the output.

### Training!

In [12]:
# While developing, it is recommended to train your model on a subset of the data... / or low epochs.
# as we didn't implemented mini-batch GD, training will be pretty slow if we update at each iteration on 60000 samples...
net.use_loss(mse, mse_grad)
epoch_num = 20 # use your own number of epochs
lr = 0.05      # use your own learning rate, I advise to play around with it and see how results change—be sure to rereun the network above across trial runs.
t1 = time.time()
net.fit(x_train, y_train, epochs=epoch_num, learning_rate=lr) # subset the data if you wish, just make sure it is a working model...
print(f"Total process time: {round(time.time() - t1,3)}")


Training epoch 1/20   error=0.017143
Training epoch 2/20   error=0.008165
Training epoch 3/20   error=0.006209
Training epoch 4/20   error=0.005093
Training epoch 5/20   error=0.004401
Training epoch 6/20   error=0.003834
Training epoch 7/20   error=0.003447
Training epoch 8/20   error=0.003121
Training epoch 9/20   error=0.002826
Training epoch 10/20   error=0.002512
Training epoch 11/20   error=0.002339
Training epoch 12/20   error=0.002209
Training epoch 13/20   error=0.002033
Training epoch 14/20   error=0.001909
Training epoch 15/20   error=0.001812
Training epoch 16/20   error=0.001736
Training epoch 17/20   error=0.001671
Training epoch 18/20   error=0.001569
Training epoch 19/20   error=0.001512
Training epoch 20/20   error=0.001446
Total process time: 640.291


### Evaluation

Exciting! Now is the time to test your model.   

    May the gradients be always in your favor.

In [13]:
output = net.predict(x_test, y_test)


Test set: Avg. loss: 0.0049, Accuracy: 9735/10000 (97%)



## 3. Benchmarking against PyTorch

How well your model performs against a similar-architecture PyTorch model?   
It is time to find out:

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset

#### Prepare the data as tensors using PyTorch DataLoader:

In [15]:
t_train =  TensorDataset(torch.Tensor(x_train),torch.Tensor(y_train))
t_test =  TensorDataset(torch.Tensor(x_test),torch.Tensor(y_test))
train_loader = torch.utils.data.DataLoader(dataset=t_train, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=t_test, batch_size=64, shuffle=False)

Define a `PyTorchNet` class with an identical architecture you used in your home-made network.

3.1 (10 pts)

In [16]:
#### SOLUTION REQUIRED  ####

class PyTorchNet(nn.Module):
    def __init__(self):
        super(PyTorchNet, self).__init__()
        input_size = 28 * 28
        num_classes = 10

        self.fc1 = nn.Linear(input_size, 128)
        self.activ1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.activ2 = nn.ReLU()
        self.fc3 = nn.Linear(64, num_classes)
        self.activ3 = nn.Tanh()


    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.activ1(self.fc1(x))
        x = self.activ2(self.fc2(x))
        x = self.activ3(self.fc3(x))

        return x

In [17]:
# Train the model
num_epochs = 20
pt_learning_rate = 0.01
pt_network = PyTorchNet()
optimizer = torch.optim.Adam(pt_network.parameters(), lr=pt_learning_rate)
criterion = nn.MSELoss()

t1 = time.time()
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        # Forward pass
        outputs = pt_network(images)
        loss = criterion(outputs, labels)
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # A handy printout:
        if (i + 1) % 500 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')
print(f"Total process time: {round(time.time() - t1,3)}")

Epoch [1/20], Step [500/938], Loss: 0.0136
Epoch [2/20], Step [500/938], Loss: 0.0093
Epoch [3/20], Step [500/938], Loss: 0.0050
Epoch [4/20], Step [500/938], Loss: 0.0113
Epoch [5/20], Step [500/938], Loss: 0.0031
Epoch [6/20], Step [500/938], Loss: 0.0039
Epoch [7/20], Step [500/938], Loss: 0.0042
Epoch [8/20], Step [500/938], Loss: 0.0015
Epoch [9/20], Step [500/938], Loss: 0.0010
Epoch [10/20], Step [500/938], Loss: 0.0040
Epoch [11/20], Step [500/938], Loss: 0.0010
Epoch [12/20], Step [500/938], Loss: 0.0120
Epoch [13/20], Step [500/938], Loss: 0.0034
Epoch [14/20], Step [500/938], Loss: 0.0030
Epoch [15/20], Step [500/938], Loss: 0.0034
Epoch [16/20], Step [500/938], Loss: 0.0027
Epoch [17/20], Step [500/938], Loss: 0.0086
Epoch [18/20], Step [500/938], Loss: 0.0034
Epoch [19/20], Step [500/938], Loss: 0.0123
Epoch [20/20], Step [500/938], Loss: 0.0096
Total process time: 107.62


Evaluation:

In [18]:
pt_network.eval()
test_losses = []
test_loss = 0
correct = 0
with torch.no_grad():
    for data, target in test_loader:
        output = pt_network(data)
        test_loss += criterion(output, target) * data.size(0)
        pred = output.data.max(1, keepdim=True)[1]
        correct += pred.eq(target.data.max(1,keepdim=True)[1]).sum()

test_loss /= len(test_loader.dataset)
test_losses.append(test_loss)
print('\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
  test_loss, correct, len(test_loader.dataset),
  100. * correct / len(test_loader.dataset)))


Test set: Avg. loss: 0.0074, Accuracy: 9571/10000 (96%)



3.2 (10 pts)

Time for some questions:
1. Which one of the models performed better? Why?
2. Which one of the models performed faster? Why?  
3. What would you change in your network's architecture?   
4. What would you change in your model's solution algorithm?

In [19]:
# check for answer 1: the same network, but with a smaller learning rate for Adam
pt_network2 = PyTorchNet()
optimizer2 = torch.optim.Adam(pt_network2.parameters(), lr=0.001)

for epoch in range(num_epochs):
    for images, labels in train_loader:
        loss = criterion(pt_network2(images), labels)
        optimizer2.zero_grad()
        loss.backward()
        optimizer2.step()

correct = 0
with torch.no_grad():
    for data, target in test_loader:
        pred = pt_network2(data).argmax(dim=1)
        correct += (pred == target.argmax(dim=1)).sum().item()
print(f"Adam with lr=0.001: {correct}/{len(test_loader.dataset)} correct")

Adam with lr=0.001: 9766/10000 correct


Write your answers here:

**1. Which one of the models performed better? Why?**

My numpy network was more accurate: about 97.35% test accuracy vs. about 95.71% for PyTorch, and its test loss was lower too. both have the same architecture (784-128-64-10 with ReLU, ReLU, tanh), so the difference comes from the training. In my network the training error goes down steadily. In PyTorch the printed loss is the loss of one batch, so it is noisy, but it also does not keep going down, it keeps jumping up and down. That looks like a step size that is too big for Adam on this problem: the weights keep jumping around the minimum instead of settling into it. The check above supports this: the same PyTorch network with lr=0.001 reached 97.66%, so the gap is about the step size and not about the framework.

**2. Which one of the models performed faster? Why?**

PyTorch was several times faster (compare the two "Total process time" lines above). My network goes over the 60,000 images one by one in a Python loop, so every epoch is 60,000 forward and backward passes. PyTorch works on mini batches of 64 images, so one epoch is only 938 steps, and each step computes the whole batch at once in optimized compiled code. The gradients (autograd) are also computed inside that compiled code, not in Python.

**3. What would you change in your network's architecture?**

* Change the output to softmax + cross-entropy instead of tanh + MSE. This is a classification problem, and cross entropy is the loss that fits classification (a surrogate loss for the 0-1 error). Its gradient with respect to the input of the softmax is simply p - y, so it does not become tiny when an output saturates.
* Make the hidden layers wider (for example 256 and 128). More neurons means more "elbows", so the network can represent more complicated functions.
* Choose the layer sizes and the number of layers as hyperparameters, using a validation set taken out of the training data, and not by looking at the test set.

I tried the first two in the Network Wars part below (together with two changes in the training), and the accuracy went up.

**4. What would you change in your model's solution algorithm?**
* Use mini batches instead of one sample per step: much faster, and each gradient is less noisy.
* Shuffle the training data at the start of every epoch because `fit` always goes over the samples in the same order..
* Make the learning rate smaller during training (big steps at the start, small steps near the end), or use an adaptive optimizer like Adam, but with a learning rate that is not too large.
* Stop the training according to the error on a validation set (early stopping) instead of a fixed number of epochs.


## 4. The Network Wars!

Here is your chance to play with your model's architecture in order to break your own benchmark set eariler.  
You can add/remove layers, play with their sizes, types, etc.   
You can add a new loss if you wish, or anything else that will fairly give your model an advantage over base.  

4.1 (15 pts)

In [20]:
#### SOLUTION REQUIRED  ####

#so what I changed:
# 1. softmax + cross entropy at the output instead of tanh + mse
# (the natural loss for classification, and the gradient does not die when the output saturates)
# 2. wider hidden layers: 256 and 128 instead of 128 and 64
# 3. shuffle the train set every epoch + make the learning rate a bit smaller every epoch

class SoftmaxLayer(Layer_Primitive):
    def forward_propagation(self, input_data):
        self.input = input_data
        e = np.exp(input_data - np.max(input_data))
        self.output = e / np.sum(e)
        return self.output

    def backward_propagation(self, output_grad, learning_rate):
        # the gradient of softmax + cross entropy together w.r.t the softmax input is (y_pred - y_true).
        # cross_entropy_grad already returns exactly that, so here we just pass it back
        return output_grad

def cross_entropy(y_true, y_pred):
    return -np.sum(y_true * np.log(y_pred + 1e-12))

def cross_entropy_grad(y_true, y_pred):
    return y_pred - y_true # works together with SoftmaxLayer


net2 = MyNetwork()
net2.add(Affine_Layer(28*28, 256))
net2.add(ActivationLayer(relu, relu_grad))
net2.add(Affine_Layer(256, 128))
net2.add(ActivationLayer(relu, relu_grad))
net2.add(Affine_Layer(128, 10))
net2.add(SoftmaxLayer())
net2.use_loss(cross_entropy, cross_entropy_grad)

epochs2 = 15
lr2 = 0.01
t1 = time.time()
for ep in range(epochs2):
    idx = np.random.permutation(len(x_train)) # new order every epoch
    print(f"epoch {ep+1}/{epochs2}, lr = {lr2:.5f}")
    net2.fit(x_train[idx], y_train[idx], epochs=1, learning_rate=lr2)
    lr2 = lr2 * 0.9
print(f"Total process time: {round(time.time() - t1,3)}")

output2 = net2.predict(x_test, y_test)

epoch 1/15, lr = 0.01000
Training epoch 1/1   error=0.210176
epoch 2/15, lr = 0.00900
Training epoch 1/1   error=0.089961
epoch 3/15, lr = 0.00810
Training epoch 1/1   error=0.058174
epoch 4/15, lr = 0.00729
Training epoch 1/1   error=0.035721
epoch 5/15, lr = 0.00656
Training epoch 1/1   error=0.020544
epoch 6/15, lr = 0.00590
Training epoch 1/1   error=0.012012
epoch 7/15, lr = 0.00531
Training epoch 1/1   error=0.007310
epoch 8/15, lr = 0.00478
Training epoch 1/1   error=0.003352
epoch 9/15, lr = 0.00430
Training epoch 1/1   error=0.001920
epoch 10/15, lr = 0.00387
Training epoch 1/1   error=0.001144
epoch 11/15, lr = 0.00349
Training epoch 1/1   error=0.000900
epoch 12/15, lr = 0.00314
Training epoch 1/1   error=0.000740
epoch 13/15, lr = 0.00282
Training epoch 1/1   error=0.000645
epoch 14/15, lr = 0.00254
Training epoch 1/1   error=0.000595
epoch 15/15, lr = 0.00229
Training epoch 1/1   error=0.000548
Total process time: 1032.32

Test set: Avg. loss: 0.0702, Accuracy: 9838/10000 

**Network Wars result**

The base network (section 2) reached 97.35% test accuracy, and the new network reached 98.38%, so the number of test mistakes dropped from 265 to 162 (almost 40% fewer). It is still a numpy only network, trained with the same one sample per step gradient descent, on the same training data, and tested on the same test set.

What changed and why:
1. **Softmax + cross entropy at the output instead of tanh + MSE.**
Softmax turns the 10 outputs into probabilities that sum to 1, and cross entropy is the natural loss for classification. The gradient of softmax and cros entropy together, with respect to the input of the softmax, is p - y (the same structure as $\hat y - y$ in Part 1, B2). So `cross_entropy_grad` returns p - y, and the `SoftmaxLayer` backward just passes it on.
2. **Wider hidden layers: 256 and 128 instead of 128 and 64.** More neurons give the network more flexibility.
3. **Shuffling every epoch.** `fit` goes over the samples in a fixed order. A new random order in every epoch makes the per sample steps closer to real stochastic gradient descent.
4. **A learning rate that gets smaller every epoch** (starts at 0.01, multiplied by 0.9 after each epoch, down to about 0.0023): bigger steps at the start, smaller steps at the end, so it does not jump around the minimum.

The price is time: every epoch took about 69 seconds instead of about 32, because the layers are wider. The test loss printed here is a cross entropy value, so it cannot be compared to the MSE of the base network. The accuracy is the fair comparison.


Download your .ipynb file and upload it to the session, then run the code chunk below and download the .html file in your submission on Moodle.